# 02 Retrieval compare (read-only, no GPU)

Reads `results/retrieval_test_*.json` from notebook 01, builds one comparison table and picks the retrieval setup to use in the full runs (notebook 04).

**Before running:** put every teammate's `retrieval_test_<name>.json` in `results/`. Missing retrievers are skipped.

Small test sets: a gap of 1 or 2 questions is noise. The notebook shows counts like `27/34` and flags ties.

## 1. Setup

In [1]:
import os, sys, json, time, random, subprocess
from pathlib import Path

REPO_URL = "https://github.com/yousefJooX/egyptian-personal-status-rag.git"
REPO_NAME = "egyptian-personal-status-rag"
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

cwd = Path.cwd()
if (cwd / "run.py").exists() or (cwd / "demo").exists():
    ROOT = cwd
else:
    ROOT = cwd / REPO_NAME
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)

DEMO = ROOT / "demo" if (ROOT / "demo").exists() else ROOT
os.chdir(DEMO)   # blocks cache indexes in data/index relative to here
for p in (str(DEMO), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=True)
print("Environment:", "Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Local")
print("ROOT:", ROOT, "| working dir:", Path.cwd())

Environment: Colab
ROOT: /content/egyptian-personal-status-rag | working dir: /content/egyptian-personal-status-rag/demo


## 2. Load test set and all retrieval results

In [ ]:
import numpy as np
import pandas as pd

def find_file(name):
    for p in (DEMO / "data" / name, ROOT / "data" / name):
        if p.exists():
            return p
    raise FileNotFoundError(f"{name} not found in {DEMO/'data'} or {ROOT/'data'}")

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

KS = (1, 3, 5, 10, 20)

def qgroup(item):
    """Question groups used in the breakdown. Adds item['qtype'] if the test set has it."""
    g = {"size": "multi-chunk" if len(item["gold_chunks"]) > 1 else "single-chunk"}
    qt = item.get("qtype") or item.get("type")
    if qt:
        g["qtype"] = qt
    return g

def score(per_q, T):
    """per_q: list of {qid, top (ranked chunk ids, up to 20)}. Answerable questions only (Task 7)."""
    ans = [p for p in per_q if T[p["qid"]]["answerable"]]
    out = {"n_answerable": len(ans)}
    for k in KS:
        rec, hit = [], 0
        for p in ans:
            gold = set(T[p["qid"]]["gold_chunks"])
            found = gold & set(p["top"][:k])
            rec.append(len(found) / len(gold))
            hit += bool(found)
        out[f"recall@{k}"] = float(np.mean(rec)) if rec else float("nan")
        out[f"hit@{k}"] = hit          # questions with at least one gold chunk in the top k
    for k in (5, 20):
        rr = []
        for p in ans:
            gold = set(T[p["qid"]]["gold_chunks"])
            ranks = [i for i, c in enumerate(p["top"][:k], 1) if c in gold]
            rr.append(1 / ranks[0] if ranks else 0.0)
        out[f"mrr@{k}"] = float(np.mean(rr)) if rr else float("nan")
    return out

test_set = load_json(find_file("test_set.json"))
T = {q["qid"]: q for q in test_set}
results_dir = DEMO / "results"
results_dir.mkdir(exist_ok=True)

ORDER = ["e5", "bge", "bm25", "hybrid", "hybrid_e5", "hybrid_rr"]
R = {}
for f in sorted(results_dir.glob("retrieval_test_*.json")):
    d = load_json(f)
    qids = {p["qid"] for p in d["per_question"]}
    if qids != set(T):
        print(f"WARNING {f.name}: question ids differ from test_set.json "
              f"({len(qids ^ set(T))} mismatched). Re-run 01 with the current test set. Skipped.")
        continue
    R[d["retriever"]] = d
names = [n for n in ORDER if n in R] + [n for n in R if n not in ORDER]
print("Found:", names)
print("Missing:", [n for n in ORDER[:5] + ["hybrid_rr"] if n not in R] or "none")
assert names, "No retrieval_test_*.json files in results/"

Found: ['e5', 'bge', 'bm25', 'hybrid', 'hybrid_e5', 'hybrid_rr']
Missing: none


## 3. Comparison table

In [3]:
S = {n: score(R[n]["per_question"], T) for n in names}
N = S[names[0]]["n_answerable"]

rows = []
for n in names:
    s = S[n]
    rows.append({"retriever": n,
                 **{f"recall@{k}": round(s[f"recall@{k}"], 3) for k in KS},
                 "hit@5": f'{s["hit@5"]}/{N}', "hit@20": f'{s["hit@20"]}/{N}',
                 "mrr@5": round(s["mrr@5"], 3),
                 "avg_seconds": round(R[n]["avg_seconds"], 3), "peak_gpu_mb": R[n]["peak_gpu_mb"]})
cmp_df = pd.DataFrame(rows)
cmp_df.to_csv(results_dir / "retrieval_compare.csv", index=False)
display(cmp_df.style.hide(axis="index"))

retriever,recall@1,recall@3,recall@5,recall@10,recall@20,hit@5,hit@20,mrr@5,avg_seconds,peak_gpu_mb
e5,0.780000,0.920000,0.920000,0.920000,0.920000,25/25,25/25,0.913000,0.019000,1606
bge,0.740000,0.880000,0.920000,0.920000,0.940000,25/25,25/25,0.910000,0.026000,4402
bm25,0.600000,0.680000,0.760000,0.840000,0.880000,20/25,24/25,0.687000,0.001000,1069
hybrid,0.740000,0.860000,0.880000,0.880000,0.940000,24/25,25/25,0.861000,0.036000,3236
hybrid_e5,0.680000,0.860000,0.860000,0.920000,0.920000,23/25,25/25,0.813000,0.014000,3258
hybrid_rr,0.900000,0.920000,0.920000,0.940000,0.940000,25/25,25/25,1.000000,0.642000,3483


## 4. Breakdown by question type

In [7]:
rows = []
for n in names:
    for gname in ("size", "qtype"):
        groups = {}
        for p in R[n]["per_question"]:
            t = T[p["qid"]]
            if t["answerable"] and gname in qgroup(t):
                groups.setdefault(qgroup(t)[gname], []).append(p)
        for gv, ps in groups.items():
            s = score(ps, T)
            rows.append({"retriever": n, "group": f"{gname}: {gv}",
                         "hit@5": f'{s["hit@5"]}/{s["n_answerable"]}', "recall@5": round(s["recall@5"], 3)})
bt = pd.DataFrame(rows)
if len(bt):
    display(bt.pivot(index="group", columns="retriever", values="hit@5")[names])
    display(bt.pivot(index="group", columns="retriever", values="recall@5")[names])
else:
    print("No groups to show.")

retriever,e5,bge,bm25,hybrid,hybrid_e5,hybrid_rr
group,,,,,,
size: multi-chunk,5/5,5/5,3/5,5/5,4/5,5/5
size: single-chunk,20/20,20/20,17/20,19/20,19/20,20/20


retriever,e5,bge,bm25,hybrid,hybrid_e5,hybrid_rr
group,,,,,,
size: multi-chunk,0.6,0.6,0.40,0.60,0.50,0.6
size: single-chunk,1.0,1.0,0.85,0.95,0.95,1.0


In [8]:
from google.colab import files
files.download("results/retrieval_compare.csv")
files.download("results/retrieval_decision.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. Head-to-head questions
Each line shows how many questions only one side finds in the top 5. A difference of 2 or less is noise.

In [5]:
def found5(n):
    return {p["qid"] for p in R[n]["per_question"]
            if T[p["qid"]]["answerable"] and set(T[p["qid"]]["gold_chunks"]) & set(p["top"][:5])}

def versus(a, b, label):
    if a not in R or b not in R:
        print(f"{label}: needs {a} and {b} (missing)"); return
    fa, fb = found5(a), found5(b)
    only_a, only_b = len(fa - fb), len(fb - fa)
    verdict = "tie within noise" if abs(only_a - only_b) <= 2 else (f"{a} better" if only_a > only_b else f"{b} better")
    print(f"{label}: {a} only {only_a} | {b} only {only_b} | both {len(fa & fb)} -> {verdict}")

versus("e5", "bge", "Best embedder        ")
versus("bge", "hybrid", "Hybrid vs dense (bge)")
versus("bm25", "bge", "BM25 vs dense        ")
versus("hybrid", "hybrid_rr", "Reranker helps?      ")
versus("hybrid_e5", "hybrid", "Hybrid: e5 vs bge    ")

if "hybrid_rr" in R and "candidates" in R["hybrid_rr"]["per_question"][0]:
    c = score([{"qid": p["qid"], "top": p["candidates"]} for p in R["hybrid_rr"]["per_question"]], T)
    print(f'\nReranker candidate pool: hit@20 = {c["hit@20"]}/{N}. '
          f'If this is low, the reranker cannot help (it only reorders these 20).')

Best embedder        : e5 only 0 | bge only 0 | both 25 -> tie within noise
Hybrid vs dense (bge): bge only 1 | hybrid only 0 | both 24 -> tie within noise
BM25 vs dense        : bm25 only 0 | bge only 5 | both 20 -> bge better
Reranker helps?      : hybrid only 0 | hybrid_rr only 1 | both 24 -> tie within noise
Hybrid: e5 vs bge    : hybrid_e5 only 0 | hybrid only 1 | both 23 -> tie within noise

Reranker candidate pool: hit@20 = 25/25. If this is low, the reranker cannot help (it only reorders these 20).


## 6. Pick the winner

In [6]:
# Rule: highest recall@5, then mrr@5, then lowest seconds.
ranked = sorted(names, key=lambda n: (-S[n]["recall@5"], -S[n]["mrr@5"], R[n]["avg_seconds"]))
best = ranked[0]
print("Ranking:", " > ".join(ranked))
print(f"Best retrieval: {best}  (hit@5 {S[best]['hit@5']}/{N}, recall@5 {S[best]['recall@5']:.3f}, mrr@5 {S[best]['mrr@5']:.3f})")

if len(ranked) > 1:
    gap = abs(S[best]["hit@5"] - S[ranked[1]]["hit@5"])
    if gap <= 2:
        print(f"Warning: {best} beats {ranked[1]} by only {gap} question(s) at hit@5. This is noise. "
              f"Prefer the simpler or faster one unless the manual check says otherwise.")

CFG = {"e5": ("e5", "dense", None), "bge": ("bge", "dense", None), "bm25": (None, "bm25", None),
       "hybrid": ("bge", "hybrid", None), "hybrid_e5": ("e5", "hybrid", None), "hybrid_rr": ("bge", "hybrid", "bge")}
emb, ret, rr = CFG.get(best, (None, None, None))
decision = {"best_retrieval": best, "embedder": emb, "retriever": ret, "reranker": rr,
            "hit@5": f"{S[best]['hit@5']}/{N}", "recall@5": round(S[best]["recall@5"], 3),
            "mrr@5": round(S[best]["mrr@5"], 3), "ranking": ranked}
save_json(results_dir / "retrieval_decision.json", decision)
print("\nSaved results/retrieval_compare.csv and results/retrieval_decision.json")
print("Use this retrieval in notebook 04 and in the LLM comparison:", decision)

Ranking: hybrid_rr > e5 > bge > hybrid > hybrid_e5 > bm25
Best retrieval: hybrid_rr  (hit@5 25/25, recall@5 0.920, mrr@5 1.000)

Saved results/retrieval_compare.csv and results/retrieval_decision.json
Use this retrieval in notebook 04 and in the LLM comparison: {'best_retrieval': 'hybrid_rr', 'embedder': 'bge', 'retriever': 'hybrid', 'reranker': 'bge', 'hit@5': '25/25', 'recall@5': 0.92, 'mrr@5': 1.0, 'ranking': ['hybrid_rr', 'e5', 'bge', 'hybrid', 'hybrid_e5', 'bm25']}
